Ваша задача — получить эмбеддинги картинки и текста, относящиеся к одной и той же сущности. 
Скачайте датасет с исходными данными (https://code.s3.yandex.net/deep-learning/t3_lesson1.zip).
У вас есть таблица data/items.csv с номером товара id, текстом text, названием картинки image_path и лейблом label в одной строке. Все картинки хранятся в отдельной директории data/images/, а названия файлов соответствуют указанным в таблице.
Напишите класс-загрузчик на базе Dataset, выполняющий токенизацию и загрузку картинок с подготовкой (масштабирование, нормализация, перевод в тензор) для выбранной модели. На вход он принимает путь до таблицы с данными data/items.csv, названия текстовой и картиночной моделей. Возвращает токенизированный текст и картинки в виде словаря с ключами image, input_ids, attention_mask, label.

In [1]:
import requests
import zipfile
import os

def download_and_extract_zip(url, extract_path, zip_filename='downloaded_file.zip'):
    """
    Скачивает ZIP-файл по URL и распаковывает его в указанную директорию.

    Параметры:
    - url (str): URL для скачивания ZIP-файла.
    - extract_path (str): путь к директории для распаковки.
    - zip_filename (str): имя файла для сохранения скачанного архива.
    """
    try:
        # Создаём директорию для распаковки, если её нет
        os.makedirs(extract_path, exist_ok=True)

        # Скачиваем файл
        print("Скачивание файла...")
        response = requests.get(url, stream=True)
        response.raise_for_status()  # Проверяем, что запрос успешен

        # Сохраняем скачанный файл
        zip_path = os.path.join(extract_path, zip_filename)
        with open(zip_path, 'wb') as file:
            for chunk in response.iter_content(chunk_size=8192):
                file.write(chunk)
        print(f"Файл успешно скачан: {zip_path}")

        # Распаковываем архив
        print("Распаковка архива...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print(f"Архив успешно распакован в: {extract_path}")

        # Удаляем ZIP-файл после распаковки (опционально)
        os.remove(zip_path)
        print("Временный ZIP-файл удалён.")

    except requests.exceptions.RequestException as e:
        print(f"Ошибка при скачивании: {e}")
    except zipfile.BadZipFile:
        print("Ошибка: файл не является ZIP-архивом или повреждён.")
    except Exception as e:
        print(f"Произошла ошибка: {e}")

# Пример использования
url = "https://code.s3.yandex.net/deep-learning/t3_lesson1.zip"  # Замените на реальную ссылку
extract_path = "data"  # Путь для распаковки
download_and_extract_zip(url, extract_path)


Скачивание файла...
Файл успешно скачан: data/downloaded_file.zip
Распаковка архива...
Архив успешно распакован в: data
Временный ZIP-файл удалён.


In [18]:
import torch
import random
import numpy as np

def set_seed(seed=42):
    """
    Устанавливает сиды для всех источников случайности в PyTorch.
    """
    # PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # Python
    random.seed(seed)
    
    # NumPy
    np.random.seed(seed)
    
    # CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Для некоторых операций — принудительно детерминистические алгоритмы
    torch.use_deterministic_algorithms(True, warn_only=True)

# Вызываем функцию в начале кода
set_seed(42)

In [15]:
import pandas as pd
df = pd.read_csv("data/data/items.csv")
df.text.tolist()

['3 лица', 'буквальный хот-дог']

In [40]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
import timm
from transformers import AutoTokenizer
from torch.nn.utils.rnn import pad_sequence


class MultimodalDataset(Dataset):
    def __init__(self, data_path, text_model, image_model, transforms=None):
        super().__init__()
        self.data_path = data_path
        self.df = pd.read_csv(f"{self.data_path}/items.csv")
        self.transforms = transforms
        self.tokenizer = AutoTokenizer.from_pretrained(text_model)
        self.encodings = self.tokenizer(self.df.text.tolist(),
                                        padding='max_length',
                                        truncation=True,
                                        return_tensors='pt'
                                        )
    

    def __len__(self):
        self.df.shape[0]


    def __getitem__(self, index):
        row = self.df.iloc[index]
        image = Image.open(f"{self.data_path}/images/{row.image_path}").convert("RGB")
        image = self.transforms(image=np.array(image))["image"]
        label = row.label
        text = row.text
        encodings = self.tokenizer(text, padding='max_length', truncation=True, return_tensors='pt')
        input_ids = encodings['input_ids'][index]
        attention_mask = encodings['attention_mask'][index]
        return image, input_ids, attention_mask, label
    

    def get_dataloader(self, *args, **kwargs):
        def collate_fn(batch):
            images = [torch.tensor(item[0]) for item in batch]
            texts = [torch.tensor(item[1]) for item in batch]
            attention_masks = [torch.tensor(item[2]) for item in batch]
            labels = torch.tensor([item[3] for item in batch])
            padded_texts = pad_sequence(texts, batch_first=True, padding_value=self.tokenizer.pad_token_id)
            return {
                "image": images,
                "input_ids": padded_texts,
                "attention_mask": attention_masks,
                "label": labels,
            }
        return DataLoader(self, collate_fn=collate_fn, *args, **kwargs)



dataset = MultimodalDataset(data_path="data/data",
                            text_model="bert-base-uncased",
                            image_model="tf_efficientnet_b0")

dataset

Допустим, вы выбрали текстовую и картиночную модели, сохранив их в переменные `text_model` и `image_model`. Теперь реализуйте аугментации для картинок с помощью `albumentations`:
1. Используйте по одной аугментации из афинных, dropout и цветовых преобразований в дополнение к масштабированию и переводу в тензор.
2. Укажите произвольные параметры аугментаций.

Напишите реализацию collate_fn, которая будет использована внутри загрузчика данных. На вход она должна принимать аугментации и токенизатор для модели, возвращать словарь с ключами `image`, `input_ids`, `attention_mask`, `label`.

In [45]:
import albumentations as albs

text_model="bert-base-uncased",
image_model="tf_efficientnet_b0"
image_model_cfg = timm.get_pretrained_cfg(image_model)
SIZE = image_model_cfg.input_size[1]


transforms = albs.Compose([
    albs.Resize(height=SIZE, width=SIZE, p=1),
    albs.Affine(rotate=(-15, 15), shear=(-10, 10), p=0.8),
    albs.GridDropout(ratio=0.5,
                     unit_size_range=(int(0.05 * SIZE), int(0.1 * SIZE)),
                     p=0.8),
    albs.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.8),
    albs.Normalize(mean=image_model_cfg.mean, std=image_model_cfg.std, p=1),
    albs.ToTensorV2(p=1),
])

dataset = MultimodalDataset(data_path="data/data",
                            text_model="bert-base-uncased",
                            image_model="tf_efficientnet_b0",
                            transforms=transforms,)

dataset[0]


(tensor([[[-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
          [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
          [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
          ...,
          [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
          [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
          [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179]],
 
         [[-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
          [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
          [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
          ...,
          [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
          [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
          [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357]],
 
         [[-1.8044, -1.8044, -1.8044,  ..., -1.8044, -1.8044, -1.8044],
          [-1.8044, -1.8044,